In [1]:
%cd ../
# Import required libraries 
from pathlib import Path
import hydra
from omegaconf import DictConfig, OmegaConf, open_dict
from hydra.core.global_hydra import GlobalHydra
import torch
from lightning import seed_everything
import matplotlib.pyplot as plt

from modules.loader.loaders import trainScaler
from cel.utils.layers import getLayers, compute_RF_numerical
from cel.utils.reporting import sorted_inputs, sorted_inputs_regression
from modules.utils.hydraqol import run_decorator

plt.rcParams.update({
    "text.usetex": True,
    "font.family": 'serif',
    'font.serif': ['computer modern roman'],
    'font.size': 20
})

d:\DSME\Code\Repos\CEL


In [2]:
#@hydra.main(config_path="../data/config", config_name="train_model", version_base="1.3")
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()
hydra.initialize(version_base=None, config_path="../data/config", job_name="test_app")

hydra.initialize()

In [ ]:
# cfg = hydra.compose(config_name="train_CE.yaml", overrides=["training=adamw_patient", "path=applewood_local", "model=InceptionTime10", "dataset=CWRU_Bearing", # BearingPD CWRU_Bearing  square-triangle_ch1_specificChannels", #midhighFreq-highFreq_ch1_specificChannels
#                                                             "path=applewood_local", "transforms=tsHFbasic", "loader=s", "seed=0",
#                                                             #"CE.loader.batch_size=20","CE.n_instances=200", 
#                                                             "CE=CENDRE", "CE.n_clusters=50", "CE.n_concepts=0", "CE.clusterMode=ACsilhouette", "CE.maskMode=gradient", "CE.domain=[ 'input','fourier']"])

cfg = hydra.compose(config_name="train_CE.yaml", overrides=["training=adamw_patient", "path=applewood_local", "model=ResNet1D18", "dataset=square-triangle_ch1_specificChannels",  #"dataset=square-triangle_ch1_specificChannels", #midhighFreq-highFreq_ch1_specificChannels
                                                            "path=applewood_local", "transforms=tsHFbasic", "loader=M", "seed=0",
                                                            "CE.loader.batch_size=20","CE.n_instances=200", 
                                                            "CE=CENDRE", "CE.n_clusters=50", "CE.n_concepts=2", "CE.clusterMode=ACsilhouette", "CE.maskMode=gradient", "CE.domain=[ 'input','fourier']"])
                                                            #"CE=ECLADts", "CE.n_concepts=2"])

# cfg = hydra.compose(config_name="train_CE.yaml", overrides=["training=adamw_patient", "path=applewood_local", "model=ResNet1D18", "dataset=ECG200",#,ECG5000,ECGFiveDays,Epilepsy,GunPoint", 
#                                                             "path=applewood_local", "transforms=tscHFNoiseWarp", "loader=xs", "seed=0",
#                                                             "CE.loader.batch_size=20","CE.n_instances=200", 
#                                                             "CE=CENDRE", "CE.n_clusters=50", "CE.n_concepts=2", "CE.clusterMode=ACsilhouette", "CE.maskMode=gradient", "CE.domain=[ 'input','fourier']"])
#                                                             #"CE=ECLADts", "CE.n_concepts=2"])

In [4]:
##############################
# Preliminaries
##############################
save_dir = Path(cfg.save_dir_model)

# Set the random seed for reproducibility
seed_everything(cfg.seed)
#torch.set_flush_denormal(True) # To avoid significant performance drop on CPU when the numbers get close to 0 (like 10e-324)

# Select the device for computation (CUDA, MPS, or CPU)
device = "cpu" if (cfg.training.device=="cuda" and torch.cuda.is_available()) else "cpu"
# device='cpu'
# Train scaler if necessary
if cfg.transforms.standardize:
    cfg.dataset.mean, cfg.dataset.std = trainScaler(cfg.dataset, cfg.loader)

Seed set to 0


In [5]:
# Instantiate transforms for training and testing
pretransform = hydra.utils.instantiate(cfg.dataset.pretransform)
train_transform = hydra.utils.instantiate(cfg.transforms.train)
valid_transform = hydra.utils.instantiate(cfg.transforms.val)
test_transform = hydra.utils.instantiate(cfg.transforms.test)

# Instantiate datasets with transforms
train_dataset = hydra.utils.instantiate(cfg.dataset.train, transform=train_transform, pretransform=pretransform, withId = True)
valid_dataset = hydra.utils.instantiate(cfg.dataset.val, transform=valid_transform, pretransform=pretransform, withId = True)
test_dataset = hydra.utils.instantiate(cfg.dataset.test, transform=test_transform, pretransform=pretransform, withId = True)

# Instantiate data loaders
# train_loader = hydra.utils.instantiate(cfg.loader.train, dataset=train_dataset)
# valid_loader = hydra.utils.instantiate(cfg.loader.val, dataset=valid_dataset)
# test_loader = hydra.utils.instantiate(cfg.loader.test, dataset=test_dataset)
train_loader, valid_loader, test_loader =  hydra.utils.instantiate(cfg.CE.loader, dataset_train=train_dataset, dataset_val=valid_dataset, dataset_test=test_dataset, task=cfg.dataset.task) 

d:\DSME\Code\Repos\CEL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [110]:
# valid_loader.pin_memory=True

In [111]:
# Instantiate model
model = hydra.utils.instantiate(cfg.model.object) #.to(device)
PATH=f'{save_dir}/best.pth'
model.load_state_dict(torch.load(PATH, weights_only=True))
getLayers(model) #, model.named_children
model.eval()

InceptionTime(
  (inception_block): InceptionBlock(
    (inception_layers): ModuleList(
      (0): InceptionModule(
        (bottleneck): NoOp()
        (convs): ModuleList(
          (0): Conv1d(1, 32, kernel_size=(39,), stride=(1,), padding=(19,), bias=False)
          (1): Conv1d(1, 32, kernel_size=(19,), stride=(1,), padding=(9,), bias=False)
          (2): Conv1d(1, 32, kernel_size=(9,), stride=(1,), padding=(4,), bias=False)
        )
        (maxconvpool): Sequential(
          (0): MaxPool1d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
          (1): Conv1d(1, 32, kernel_size=(1,), stride=(1,), bias=False)
        )
        (bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): ReLU()
      )
      (1-9): 9 x InceptionModule(
        (bottleneck): Conv1d(128, 32, kernel_size=(1,), stride=(1,), bias=False)
        (convs): ModuleList(
          (0): Conv1d(32, 32, kernel_size=(39,), stride=(1,), padding=(19,), bi

In [112]:
# Instantiate CE object
CE=hydra.utils.instantiate(cfg.CE.object, model=model)

In [113]:
# Fit CE object if it has not alerady been fit
#cfg.mode='force' # comment out
fitExists, reportExists = CE.check_existence(cfg.CE)
if cfg.mode == "force" or not fitExists:
    print("training Explainer")
    CE.fit(train_loader, cfg.CE, device=device)
    CE.save(cfg.CE)
    # Save the configuration
    with open(f"{cfg.CE.save_dir}/config.yaml", "w") as f:
        OmegaConf.save(config=cfg, f=f)
else:
    CE.load(cfg.CE, device=device)
    print("loaded Explainer from files")

loaded Explainer from files


In [114]:
# batchx, batchy,batchid = next(iter(valid_loader))
# ma = CE.eval(batchx)
if cfg.CE.name == 'ConceptShap':
    c_indexes = [int(k) for k, _ in sorted(CE.importances.items(), key=lambda item: item[1], reverse=True)]
else:
    c_indexes=None

c_indexes=[0, 1]

In [ ]:
# Make CE report if it does not exist already
# cfg.CE.decorrelated=True
# cfg.CE.domain='input'
cfg.mode='force-report'
ds_fs = getattr(cfg.dataset, 'fs', None) or 1
if  cfg.mode == "force" or cfg.mode == "force-report" or not reportExists:
    if cfg.dataset.task == "classification":
        batch = sorted_inputs(
            CE.model, valid_loader, n_classes=cfg.dataset.n_classes, n_repeats=1
        )
    elif cfg.dataset.task == "regression" and cfg.transforms.standardize:
        batch = sorted_inputs_regression(
            valid_loader, partitions=5,samples_per_partition=1, out_range=[-2,2],int_out=cfg.CE.multilabel_out, astensor=True
        )
    else:
        batch = next(iter(valid_loader))

    if isinstance(batch, dict):
        test_inputs, test_truth, test_id = batch["x"][:4], batch["y"][:4], batch["id"][:4]
    else:
        test_inputs, test_truth, test_id = batch[:4]

    for domain in cfg.CE.domain:
        print(f'Plotting report on {domain} domain')
        test_masks, test_outputs = CE.eval(test_inputs,domain=domain, device=device, outdevice='cpu', maskMode=getattr(cfg.CE, 'maskMode', 'none'), noisetunnel=False)
        # c_indexes=[int(k) for k, v in sorted(EC.importances.items(), key=lambda item: item[1],reverse=True)]

        CE.report(
            test_masks,
            cfg.CE,
            test_inputs,
            test_outputs,
            domain=domain,
            fs=ds_fs,
            test_truth=test_truth,
            c_indexes=c_indexes,
            DS_cfg=cfg.dataset,
            font_base = 5,
            destandardize=cfg.transforms.standardize,
            visualization_mode='perceptual',
            classwise_IS=True,
            use_clamp=False,
            importance_thr=0.5,
            show=True
        )

In [ ]:
# Plot ground-truth primitive masks (synthetic datasets only)
from datasets import Dataset
from cel.utils.reporting import plot_primitives

maskcols = list(getattr(cfg.dataset, 'maskcols', []) or [])
if not maskcols:
    print(f"dataset {cfg.dataset.name} has no maskcols; skipping primitives plot")
else:
    # 1) keep the bare union columns; the _<xcol> per-channel columns duplicate them
    xcols = list(cfg.dataset.xcols)
    keep = [c for c in maskcols if not any(c.endswith(f'_{xc}') for xc in xcols)] or maskcols

    # 2) reload the raw dataset (the loader strips maskcols) and select the report rows
    ds = Dataset.load_from_disk(cfg.dataset.val.dataset.dataset_path)
    rows = test_id.cpu().long().tolist()
    n_ch = test_inputs.shape[1]

    # 3) assemble {name: (n, C, L)}; single-channel masks broadcast over channels
    primitives = {}
    for col in keep:
        m = torch.tensor(ds[col], dtype=torch.float32)[rows]
        if m.dim() == 2:
            m = m.unsqueeze(1)
        if m.shape[1] == 1 and n_ch > 1:
            m = m.expand(-1, n_ch, -1)
        primitives[col] = m

    plot_primitives(primitives, test_inputs, save_path=cfg.CE.save_dir, fs=ds_fs,
                    test_truth=test_truth, DS_cfg=cfg.dataset,
                    destandardize=cfg.transforms.standardize, font_base=5, show=True)

In [116]:
# if 'fourier' in cfg.CE.domain:
#     # Reload CENDRE so edits to plot_CC pick up without a kernel restart
#     import importlib, cel.CE.CENDRE as _cendre_mod
#     importlib.reload(_cendre_mod)
#     CE.__class__ = _cendre_mod.CENDRE

#     cc=CE.concept_correspondence(valid_loader, num_batches=10, device=device)
#     CE.plot_CC(cc,cfg.CE, fs=ds_fs, show=True, horizontal=True, DS_cfg=cfg.dataset)